In [1]:
#inception_v3
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import inception_v3
import os



# Choose dataset: 'MNIST', 'CIFAR10', 'FashionMNIST'
dataset_name = 'MNIST'

# Transform based on the chosen dataset
if dataset_name == 'MNIST'  :
    transform = transforms.Compose([
        transforms.Resize(299),
        transforms.Grayscale(num_output_channels=3),
        transforms.CenterCrop(299),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
if dataset_name == 'FashionMNIST':
  transform = transforms.Compose([transforms.Grayscale(num_output_channels=3),transforms.Resize((299,299)),
                    transforms.ToTensor(),
                    transforms.Normalize((0.1307,), (0.3081,))])
if dataset_name == 'CIFAR10':
  transform =transforms.Compose([

        transforms.Resize(299),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])



normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
# Load dataset
if dataset_name == 'MNIST':
    trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
elif dataset_name == 'CIFAR10':
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True,transform=transform)
else:
    trainset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the pre-trained Inception_v3 model
model = inception_v3(pretrained=True)
num_classes=10
model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)
# model.load_state_dict(torch.load('inception_v3_MNIST.pth'))

C:\Users\d4050\.conda\envs\torch\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\d4050\.conda\envs\torch\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [2]:
model.load_state_dict(torch.load('model/inception_v3_MNIST_best.pth'))
model = model.to(device)

In [3]:
def pgd(model, x, y, epsilon, alpha, num_iter):
    # Construct PGD adversarial examples on the examples X
    # inputs:
    #     net: the network through which we pass the inputs
    #     x: the original example which we aim to perturb to make an adversarial example
    #     y: the true label of x
    #     alpha: step size
    #     epsilon: perturbation budget
    #     iter: number of iterations in the PGD algorithm
    # test
    # outputs:
    #     x_adv : the adversarial example constructed from x
    delta = torch.zeros_like(x, requires_grad=True)
    for t in range(num_iter):
        loss = nn.CrossEntropyLoss()(model(x + delta), y)
        loss.backward()
        delta.data = (delta + x.shape[0] * alpha * delta.grad.data).clamp(-epsilon, epsilon)
        delta.grad.zero_()

    return torch.clip(delta.detach() + x, 0, 1)

def fgsm(model, x, y, epsilon):
    # Construct FGSM adversarial examples on the examples X
    #     inputs:
    #         net: the network through which we pass the inputs
    #         x: the original example which we aim to perturb to make an adversarial example
    #         y: the true label of x
    #         eps: perturbation budget
    #     outputs:
    #         x_adv : the adversarial example constructed from x

    eta = torch.zeros_like(x, requires_grad=True)
    loss = nn.CrossEntropyLoss()(model(x + eta), y)
    loss.backward()
    return torch.clip(epsilon * eta.grad.detach().sign() + x, 0, 1)

# metric = PeakSignalNoiseRatio()

eps = 8.0/255
alpha = 5.0
its = 7

In [4]:
from tqdm import tqdm
atk_img = []
atk_label = []
model.eval()
val_correct = 0
# val_bar = tqdm(trainloader, position=0, leave=True)
val_bar = tqdm(testloader, position=0, leave=True)
for x, y in val_bar:
    x, y = x.to(device), y.to(device)
    # atk_x = fgsm(model, x, y, eps)
    atk_x = pgd(model, x, y, eps, alpha, its)
    # print(atk_x.shape)
    # break
    # atk_x = atk_x.reshape(-1, 28, 28)
    atk_img += atk_x
    atk_label += y
    # with torch.no_grad():
    #     y_pred = model(atk_x)
    #     atk_x = atk_x.reshape(-1, 28, 28)
    #     atk_img += atk_x
    #     atk_label += y
    #     class_pred = y_pred.argmax(dim=1)
    #     val_correct += class_pred.eq(y).sum().item()
# val_accuracy = val_correct / len(testset)

# print(f" attack accuracy: {val_accuracy:.4f}")

 72%|███████▏  | 224/313 [10:45<04:16,  2.88s/it]


KeyboardInterrupt: 

In [ ]:
print(len(atk_label))

In [ ]:
from torch.utils.data import Dataset
class AdvDataset(Dataset):
    def __init__(self, x, y):
        self.x = x.copy()
        self.y = y.copy()

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.x)

In [ ]:
adv_dataset = AdvDataset(atk_img, atk_label)
torch.save(adv_dataset, './adv_data/inc_pgd_mnist_test')